In [1]:
%load_ext autoreload
%autoreload 2

In [44]:
from collections import defaultdict
import copy
import os
import pprint
import functools
from pathlib import Path

import hydra
import duckdb
from omegaconf import OmegaConf
from einops import rearrange
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches
import matplotlib.colors
import seaborn as sns
font_scale = 7
sns.set_theme(style='ticks', font_scale=font_scale, palette=sns.color_palette('Set2'),)
import sqlalchemy as sa
import marimo as mo
import seaborn as sns
import lightning.pytorch
import polars as pl
from tqdm import tqdm
import torch
from great_tables import GT

from conf import conf
from dafm import datasets, models, plots, utils

In [3]:
duckdb.sql("""
attach '../runs.sqlite';
use runs;
""")

In [4]:
engine = conf.get_engine()
session = conf.sa.orm.Session(engine)
session.begin()

# Run queries

### Datasets

In [5]:
dataset_cols = """
    Dataset.id,
    time_step_count,
    time_step_count_drop_first,
    observe_every_n_time_steps,
"""

In [6]:
dataset_info = pl.DataFrame(dict(
    dataset_name=['Lorenz96', 'KuramotoSivashinsky', 'NavierStokesDim64', 'NavierStokesDim256'],
    dataset_name_latex=["Lorenz '96", 'Kuramoto-Sivashinsky (1,024 dim)', r'Navier-Stokes ($64 \times 64$)', r'Navier-Stokes ($256 \times 256$)'],
    multiple=1,
))
dataset_info

dataset_name,dataset_name_latex,multiple
str,str,i32
"""Lorenz96""","""Lorenz '96""",1
"""KuramotoSivashinsky""","""Kuramoto-Sivashinsky (1,024 di…",1
"""NavierStokesDim64""","""Navier-Stokes ($64 \times 64$)""",1
"""NavierStokesDim256""","""Navier-Stokes ($256 \times 256…",1


##### Lorenz '96

In [7]:
l96_rows = duckdb.sql(f"""
    select
        {dataset_cols}
        dataset_name,
        dataset_name_latex,
    from dataset_info
    join Dataset
    on dataset_name = 'Lorenz96'
    join Lorenz96 on Dataset.id = Lorenz96.id
    where true
    and state_dimension = 1e6
    and use_predicted_state_perturbation = true
    and predicted_state_perturbation_std = 1e-2
""").pl()
assert len(l96_rows) == 1

##### Kuramoto-Sivashinsky

In [8]:
ks_rows = duckdb.sql(f"""
    select
        {dataset_cols}
        dataset_name,
        dataset_name_latex,
    from dataset_info
    join Dataset
    on dataset_name = 'KuramotoSivashinsky'
    join KuramotoSivashinsky on Dataset.id = KuramotoSivashinsky.id
    where true
    and state_dimension = 1024
    and floating_point_precision = 32
    and use_predicted_state_perturbation = true
    and predicted_state_perturbation_std = 1e-2
""").pl()
assert len(ks_rows) == 1

##### Navier-Stokes ($64 \times 64$)

In [9]:
ns64_rows = duckdb.sql(f"""
    select
        {dataset_cols}
        dataset_name,
        dataset_name_latex,
    from dataset_info
    join Dataset
    on dataset_name = 'NavierStokesDim64'
    join NavierStokes on Dataset.id = NavierStokes.id
    where true
    and state_dimension = 3*64*64
    and use_predicted_state_perturbation = true
    and predicted_state_perturbation_std = 1e-2
""").pl()
assert len(ns64_rows) == 1

##### Navier-Stokes ($256 \times 256$)

In [10]:
ns256_rows = duckdb.sql(f"""
    select
        {dataset_cols}
        dataset_name,
        dataset_name_latex,
    from dataset_info
    join Dataset
    on dataset_name = 'NavierStokesDim256'
    join NavierStokes on Dataset.id = NavierStokes.id
    where true
    and state_dimension = 3*256*256
    and time_step_count = 6000
    and time_step_size = 1e-4
    and observe_every_n_time_steps = 100
    and use_predicted_state_perturbation = true
    and predicted_state_perturbation_std = 1e-2
""").pl()
assert len(ns256_rows) == 1

### Models

In [11]:
sampling_time_step_counts = (5, 10, 20, 50, 100)

In [12]:
var = {
    'EnSF': [
        ['epsilon_alpha', r'$\epsilon_{\alpha}$'],
        ['epsilon_beta', r'$\epsilon_{\beta}$'],
    ],
    'EnFF-OT': [
        ['log10(sigma_min)', r'$\log_{10}(\sigma_{\min})$'],
        ['lambda', r'$\lambda$'],
    ],
}
var['EnFF-F2P'] = var['EnFF-OT']

##### EnsF

In [13]:
ensf_hyperparameters = pl.DataFrame(dict(
    dataset_name=['Lorenz96', 'KuramotoSivashinsky', 'NavierStokesDim64', 'NavierStokesDim256'],
    epsilon_alpha=[1.0, 1.0, 1.0, 1.0],
    epsilon_beta=[0.275, 0.275, 0.275, 0.275],
))

In [14]:
label = 'EnSF'
var1, var1_latex = var[label][0]
var2, var2_latex = var[label][1]
ensf_rows = duckdb.sql(f"""
    select
        Model.id,
        {label!r} as label,
        {var1} as var1,
        {var2} as var2,
        sampling_time_step_count,
    from Model
    join ScoreMatchingMarginal on Model.id = ScoreMatchingMarginal.id
    join Bao2024EnsembleScoreMatching on ScoreMatchingMarginal.DiffusionPath = Bao2024EnsembleScoreMatching.id
    where true
    and sampling_time_step_count in {sampling_time_step_counts}
    and (epsilon_alpha, epsilon_beta) in (select epsilon_alpha, epsilon_beta from ensf_hyperparameters)
""")
ensf_multiple = (
    len(sampling_time_step_counts)
    * len(ensf_hyperparameters[['epsilon_alpha', 'epsilon_beta']].unique())
)
assert len(ensf_rows) == ensf_multiple

##### EnFF-OT

In [15]:
enff_ot_hyperparameters = pl.DataFrame(dict(
    dataset_name=['Lorenz96', 'KuramotoSivashinsky', 'NavierStokesDim64', 'NavierStokesDim256'],
    sigma_min=[0.1, 1e-05, 1e-05, 0.01],
    constant=[0.05, 0.05, 0.05, 0.005],
))

In [16]:
label = 'EnFF-OT'
var1, var1_latex = var[label][0]
var2, var2_latex = var[label][1]
enff_ot_rows = duckdb.sql(f"""
    select
        Model.id,
        {label!r} as label,
        {var1} as var1,
        constant as var2,
        sampling_time_step_count,
    from Model
    join FlowMatchingMarginal on Model.id = FlowMatchingMarginal.id
    join ConditionalOptimalTransport on FlowMatchingMarginal.DiffusionPath = ConditionalOptimalTransport.id
    join Local on FlowMatchingMarginal.EnergyGuidance = Local.id
    join Constant on Local.Schedule = Constant.id
    where true
    and sampling_time_step_count in {sampling_time_step_counts}
    and (sigma_min, constant) in (select sigma_min, constant from enff_ot_hyperparameters)
""")
enff_ot_multiple = (
    len(sampling_time_step_counts)
    * len(enff_ot_hyperparameters[['sigma_min', 'constant']].unique())
)
assert len(enff_ot_rows) == enff_ot_multiple

##### EnFF-F2P

In [17]:
enff_f2p_hyperparameters = pl.DataFrame(dict(
    dataset_name=['Lorenz96', 'KuramotoSivashinsky', 'NavierStokesDim64', 'NavierStokesDim256'],
    sigma_min=[0.01, 0.001, 1e-05, 0.001],
    constant=[0.05, 0.005, 0.005, 0.001],
))

In [18]:
label = 'EnFF-F2P'
var1, var1_latex = var[label][0]
var2, var2_latex = var[label][1]
enff_f2p_rows = duckdb.sql(f"""
    select
        Model.id,
        {label!r} as label,
        {var1} as var1,
        constant as var2,
        sampling_time_step_count,
    from Model
    join FlowMatchingMarginal on Model.id = FlowMatchingMarginal.id
    join PreviousPosteriorToPredictive on FlowMatchingMarginal.DiffusionPath = PreviousPosteriorToPredictive.id
    join Local on FlowMatchingMarginal.EnergyGuidance = Local.id
    join Constant on Local.Schedule = Constant.id
    where true
    and sampling_time_step_count in {sampling_time_step_counts}
    and (sigma_min, constant) in (select sigma_min, constant from enff_f2p_hyperparameters)
""")
enff_f2p_multiple = (
    len(sampling_time_step_counts)
    * len(enff_f2p_hyperparameters[['sigma_min', 'constant']].unique())
)
assert len(enff_f2p_rows) == enff_f2p_multiple

### General

In [19]:
rows = duckdb.sql(f"""
    select
        alt_id,
        model_rows.*,
        dataset_rows.*,
        rng_seed,
    from Conf
    join (
        select * from l96_rows
        union
        select * from ks_rows
        union
        select * from ns64_rows
        union
        select * from ns256_rows
    ) as dataset_rows on Conf.Dataset = dataset_rows.id
    join (
        select * from ensf_rows
        union
        select * from enff_ot_rows
        union
        select * from enff_f2p_rows
    ) as model_rows on Conf.Model = model_rows.id
    where true
    and rng_seed in (462133975, 979497033, 97616566, 715319214, 19704671)
""")
rng_seed_multiple = 5
dataset_multiple = dataset_info['multiple'].sum()
model_multiple = 3
multiple = rng_seed_multiple * dataset_multiple * model_multiple * len(sampling_time_step_counts)
assert len(rows) == multiple, f'{len(rows) = } != {multiple}'

In [20]:
filepaths = duckdb.sql("""
select format('~/out/dafm/runs/{}/dataset_metrics.csv', alt_id) as filepath from rows
""").pl()
filepath_exists = []
for f in filepaths.get_column('filepath'):
    f = Path(f).expanduser()
    exists = f.exists()
    if not exists:
        print(f"File path does not exist: '{f}'")
    filepath_exists.append(exists)
filepaths = pl.DataFrame(dict(filepath=filepaths.get_column('filepath'), exists=filepath_exists))
assert filepaths['exists'].any()
duckdb.sql("""
set variable dataset_metrics_filepaths = (
    select list(filepath) from filepaths where exists
)
""")

File path does not exist: '/home/ttransue/out/dafm/runs/p2ksbln1/dataset_metrics.csv'
File path does not exist: '/home/ttransue/out/dafm/runs/hupo7lqb/dataset_metrics.csv'
File path does not exist: '/home/ttransue/out/dafm/runs/osp7mzmm/dataset_metrics.csv'
File path does not exist: '/home/ttransue/out/dafm/runs/n799vr6p/dataset_metrics.csv'
File path does not exist: '/home/ttransue/out/dafm/runs/wz5he02k/dataset_metrics.csv'


##### Time (seconds)

In [21]:
logged_metrics = duckdb.sql(f"""
    select rows.*, logs.*,
    from (
        select split(filename, '/')[-2] as alt_id, step, time_s, crps, rmse,
        from read_csv(getvariable(dataset_metrics_filepaths), filename=true, union_by_name=true)
    ) as logs
    join rows on rows.alt_id = logs.alt_id
    where true
    and (logs.step - time_step_count_drop_first - 1) % observe_every_n_time_steps == 0 -- include only analysis time steps
""")
logged_metrics.show(max_width=100)

┌──────────┬───────┬──────────┬───┬────────────────────┬─────────────────────┐
│  alt_id  │  id   │  label   │ … │        crps        │        rmse         │
│ varchar  │ int64 │ varchar  │   │       double       │       double        │
├──────────┼───────┼──────────┼───┼────────────────────┼─────────────────────┤
│ ejww7pqp │   923 │ EnFF-OT  │ … │  8.225206608025655 │  0.2789247588606888 │
│ ejww7pqp │   923 │ EnFF-OT  │ … │ 10.520804785557536 │  0.3373876881970976 │
│ ejww7pqp │   923 │ EnFF-OT  │ … │ 13.802430712506379 │  0.4319785871259978 │
│ ejww7pqp │   923 │ EnFF-OT  │ … │  16.23088074844767 │  0.5123434736593424 │
│ ejww7pqp │   923 │ EnFF-OT  │ … │ 18.256828012898126 │  0.5711332885505038 │
│ ejww7pqp │   923 │ EnFF-OT  │ … │ 20.112169497312756 │  0.6295765471075545 │
│ ejww7pqp │   923 │ EnFF-OT  │ … │ 21.729935338715585 │  0.6790665694263542 │
│ ejww7pqp │   923 │ EnFF-OT  │ … │  22.54841623383221 │  0.7088145445677171 │
│ ejww7pqp │   923 │ EnFF-OT  │ … │  23.689419238544

In [34]:
max_observation_steps_back = duckdb.sql(f"""
    select
        dataset_name,
        max(observation_step_count) as observation_step_count,
    from (
        select
            dataset_name,
            count(*) as observation_step_count,
        from logged_metrics
        group by alt_id, dataset_name
    )
    group by dataset_name
""")
failed_before_finish = duckdb.sql(f"""
    select
        observation_steps_back.alt_id,
        observation_steps_back.dataset_name,
        observation_steps_back.label,
        observation_steps_back.observation_step_count,
    from (
        select
            alt_id,
            dataset_name,
            label,
            count(*) as observation_step_count,
        from logged_metrics
        group by alt_id, dataset_name, label
    ) as observation_steps_back
    join max_observation_steps_back
    on max_observation_steps_back.dataset_name = observation_steps_back.dataset_name
    and observation_steps_back.observation_step_count < max_observation_steps_back.observation_step_count
""")
# max_observation_steps_back
failed_before_finish
# assert len(logged_metrics) > 0 and len(failed_before_finish) == 0, failed_before_finish

┌──────────┬────────────────────┬──────────┬────────────────────────┐
│  alt_id  │    dataset_name    │  label   │ observation_step_count │
│ varchar  │      varchar       │ varchar  │         int64          │
├──────────┼────────────────────┼──────────┼────────────────────────┤
│ eodrjd2l │ NavierStokesDim256 │ EnSF     │                      1 │
│ k4spmxbw │ NavierStokesDim64  │ EnFF-OT  │                     20 │
│ rji59uui │ NavierStokesDim256 │ EnSF     │                      1 │
│ hvkqkhvy │ NavierStokesDim64  │ EnFF-OT  │                     20 │
│ 1dykblcc │ NavierStokesDim64  │ EnFF-F2P │                    140 │
│ 2jri38f4 │ NavierStokesDim64  │ EnFF-OT  │                     20 │
│ anubpxr7 │ NavierStokesDim64  │ EnFF-F2P │                    130 │
│ rthzxiyo │ NavierStokesDim256 │ EnSF     │                      1 │
│ v89744x7 │ NavierStokesDim64  │ EnFF-OT  │                     10 │
│ p96e71ql │ NavierStokesDim64  │ EnFF-F2P │                     20 │
│ y1aqmsfu │ NavierS

In [35]:
logged_metrics = duckdb.sql("""
    select *
    from logged_metrics
    where alt_id not in (select alt_id from failed_before_finish)
""")

In [36]:
group_by = """
    alt_id,
    rng_seed,
    dataset_name,
    dataset_name_latex,
    label,
    sampling_time_step_count,
"""
logged_metrics_means = duckdb.sql(f"""
    select
        {group_by}
        mean(time_s) as time_s_mean,
        mean(rmse) as rmse_mean,
        mean(crps) as crps_mean,
    from logged_metrics
    group by
        {group_by}
""")
logged_metrics_means.show(max_width=100)

┌──────────┬───────────┬───┬──────────────────────┬──────────────────────┬────────────────────┐
│  alt_id  │ rng_seed  │ … │     time_s_mean      │      rmse_mean       │     crps_mean      │
│ varchar  │   int64   │   │        double        │        double        │       double       │
├──────────┼───────────┼───┼──────────────────────┼──────────────────────┼────────────────────┤
│ 3wnjb5aj │  97616566 │ … │   0.3717046352333198 │   4.1968134562174475 │  1560.571221923828 │
│ z2z9xpre │ 715319214 │ … │  0.12512846293333435 │  0.49985928695338466 │ 218.84242171446482 │
│ 5fu88tf6 │  19704671 │ … │    0.273253505455001 │  0.11245411485433579 │ 11.961837072372436 │
│ u6jqocua │ 715319214 │ … │   1.1410480688374958 │   1.8644245952367782 │ 1376.7967880249023 │
│ yg6i3uu0 │ 979497033 │ … │   0.6993778044749999 │  0.06559471793472767 │  6.958630033731461 │
│ 2y0609mw │ 715319214 │ … │    7.156780731412494 │   0.3345907345414162 │  319.1608522415161 │
│ q7lipbnd │ 715319214 │ … │   0.5693456

In [40]:
group_by = """
    dataset_name,
    dataset_name_latex,
    label,
    sampling_time_step_count,
"""
table_stats = duckdb.sql(f"""
    select
        {group_by}
        mean(time_s_mean) as time_s_mean,
        stddev_pop(time_s_mean) as time_s_std,
        mean(rmse_mean) as rmse_mean,
        stddev_pop(rmse_mean) as rmse_std,
    from logged_metrics_means
    group by
        {group_by}
""")
table_stats.show(max_width=100)

┌─────────────────────┬──────────────────────┬───┬─────────────────────┬──────────────────────┐
│    dataset_name     │  dataset_name_latex  │ … │      rmse_mean      │       rmse_std       │
│       varchar       │       varchar        │   │       double        │        double        │
├─────────────────────┼──────────────────────┼───┼─────────────────────┼──────────────────────┤
│ Lorenz96            │ Lorenz '96           │ … │  0.3238819556683302 │  0.0004141803813092… │
│ NavierStokesDim64   │ Navier-Stokes ($64…  │ … │ 0.16436499262601134 │ 0.006584504688102703 │
│ Lorenz96            │ Lorenz '96           │ … │    1.39022555410862 │  0.0003868455766500… │
│ KuramotoSivashinsky │ Kuramoto-Sivashins…  │ … │   1.704363520428019 │ 0.011422844720369581 │
│ NavierStokesDim64   │ Navier-Stokes ($64…  │ … │ 0.21773426364362242 │  0.00569175719446883 │
│ KuramotoSivashinsky │ Kuramoto-Sivashins…  │ … │ 0.19994535018656712 │  0.0003150283872424… │
│ Lorenz96            │ Lorenz '96      

In [41]:
table_stats = duckdb.sql(f"""
    select
        *,
        time_s_mean = min(time_s_mean) over (partition by dataset_name, sampling_time_step_count) as is_lowest_time,
        rmse_mean = min(rmse_mean) over (partition by dataset_name, sampling_time_step_count) as is_lowest_rmse,
    from table_stats
""")
table_stats.show(max_width=100)

┌─────────────────────┬──────────────────────┬──────────┬───┬────────────────┬────────────────┐
│    dataset_name     │  dataset_name_latex  │  label   │ … │ is_lowest_time │ is_lowest_rmse │
│       varchar       │       varchar        │ varchar  │   │    boolean     │    boolean     │
├─────────────────────┼──────────────────────┼──────────┼───┼────────────────┼────────────────┤
│ Lorenz96            │ Lorenz '96           │ EnFF-F2P │ … │ false          │ false          │
│ Lorenz96            │ Lorenz '96           │ EnSF     │ … │ true           │ false          │
│ Lorenz96            │ Lorenz '96           │ EnFF-OT  │ … │ false          │ true           │
│ KuramotoSivashinsky │ Kuramoto-Sivashins…  │ EnSF     │ … │ false          │ false          │
│ KuramotoSivashinsky │ Kuramoto-Sivashins…  │ EnFF-OT  │ … │ true           │ false          │
│ KuramotoSivashinsky │ Kuramoto-Sivashins…  │ EnFF-F2P │ … │ false          │ true           │
│ NavierStokesDim64   │ Navier-Stokes ($

In [42]:
def join(t):
    if t != '':
        t = f'{t}.'
    return f'{t}dataset_name, {t}sampling_time_step_count, {t}label'
table = duckdb.sql(rf"""
    select
        table_stats.label as 'Model',
        table_stats.dataset_name_latex as 'System',
        table_stats.sampling_time_step_count as '$T$',
        if(is_lowest_time, format('$\bm{{{{{{}}}}}}$', fmt_stats.time_s), format('${{}}$', fmt_stats.time_s)) as 'Time (s)',
        if(is_lowest_rmse, format('$\bm{{{{{{}}}}}}$', fmt_stats.rmse), format('${{}}$', fmt_stats.rmse)) as 'RMSE',
    from table_stats
    join (
        select
            {join('')},
            format('{{{{{{:.3f}}}}}}_{{{{\pm {{:.3f}}}}}}', time_s_mean, time_s_std) as time_s,
            format('{{{{{{:.3f}}}}}}_{{{{\pm {{:.3f}}}}}}', rmse_mean, rmse_mean) as rmse,
        from table_stats
    ) as fmt_stats
    on ({join('table_stats')}) = ({join('fmt_stats')})
    order by
        {join('table_stats')} desc
""")
table

┌──────────┬──────────────────────────────────┬───────┬────────────────────────────┬────────────────────────────┐
│  Model   │              System              │  $T$  │          Time (s)          │            RMSE            │
│ varchar  │             varchar              │ int64 │          varchar           │          varchar           │
├──────────┼──────────────────────────────────┼───────┼────────────────────────────┼────────────────────────────┤
│ EnSF     │ Kuramoto-Sivashinsky (1,024 dim) │     5 │ ${0.021}_{\pm 0.013}$      │ ${1.630}_{\pm 1.630}$      │
│ EnFF-OT  │ Kuramoto-Sivashinsky (1,024 dim) │     5 │ $\bm{{0.020}_{\pm 0.011}}$ │ ${0.082}_{\pm 0.082}$      │
│ EnFF-F2P │ Kuramoto-Sivashinsky (1,024 dim) │     5 │ ${0.022}_{\pm 0.014}$      │ $\bm{{0.065}_{\pm 0.065}}$ │
│ EnSF     │ Kuramoto-Sivashinsky (1,024 dim) │    10 │ $\bm{{0.060}_{\pm 0.023}}$ │ ${0.640}_{\pm 0.640}$      │
│ EnFF-OT  │ Kuramoto-Sivashinsky (1,024 dim) │    10 │ ${0.066}_{\pm 0.017}$      │ $\b

In [50]:
table_pl.pivot(
    on=['System'],
    index=['Model', '$T$'],
    values=['Time (s)', 'RMSE'],
    separator='|',
)

Model,$T$,"Time (s)|Kuramoto-Sivashinsky (1,024 dim)",Time (s)|Lorenz '96,Time (s)|Navier-Stokes ($256 \times 256$),Time (s)|Navier-Stokes ($64 \times 64$),"RMSE|Kuramoto-Sivashinsky (1,024 dim)",RMSE|Lorenz '96,RMSE|Navier-Stokes ($256 \times 256$),RMSE|Navier-Stokes ($64 \times 64$)
str,i64,str,str,str,str,str,str,str,str
"""EnSF""",5,"""${0.021}_{\pm 0.013}$""","""$\bm{{0.260}_{\pm 0.010}}$""",null,null,"""${1.630}_{\pm 1.630}$""","""${7.674}_{\pm 7.674}$""",null,null
"""EnFF-OT""",5,"""$\bm{{0.020}_{\pm 0.011}}$""","""${0.277}_{\pm 0.053}$""","""$\bm{{0.124}_{\pm 0.004}}$""","""$\bm{{0.237}_{\pm 0.017}}$""","""${0.082}_{\pm 0.082}$""","""$\bm{{0.624}_{\pm 0.624}}$""","""${0.495}_{\pm 0.495}$""","""${0.164}_{\pm 0.164}$"""
"""EnFF-F2P""",5,"""${0.022}_{\pm 0.014}$""","""${0.273}_{\pm 0.049}$""","""${0.125}_{\pm 0.009}$""","""${0.258}_{\pm 0.033}$""","""$\bm{{0.065}_{\pm 0.065}}$""","""${0.649}_{\pm 0.649}$""","""$\bm{{0.133}_{\pm 0.133}}$""","""$\bm{{0.067}_{\pm 0.067}}$"""
"""EnSF""",10,"""$\bm{{0.060}_{\pm 0.023}}$""","""$\bm{{0.553}_{\pm 0.016}}$""","""${0.221}_{\pm 0.076}$""","""${0.325}_{\pm 0.025}$""","""${0.640}_{\pm 0.640}$""","""${3.299}_{\pm 3.299}$""","""${5.405}_{\pm 5.405}$""","""${1.560}_{\pm 1.560}$"""
"""EnFF-OT""",10,"""${0.066}_{\pm 0.017}$""","""${0.581}_{\pm 0.109}$""","""${0.210}_{\pm 0.080}$""","""$\bm{{0.298}_{\pm 0.025}}$""","""$\bm{{0.056}_{\pm 0.056}}$""","""${0.583}_{\pm 0.583}$""","""${0.497}_{\pm 0.497}$""","""${0.112}_{\pm 0.112}$"""
…,…,…,…,…,…,…,…,…,…
"""EnFF-OT""",50,"""$\bm{{0.248}_{\pm 0.093}}$""","""${3.133}_{\pm 0.500}$""","""$\bm{{0.512}_{\pm 0.023}}$""","""$\bm{{0.463}_{\pm 0.049}}$""","""${1.571}_{\pm 1.571}$""","""${0.706}_{\pm 0.706}$""","""${0.376}_{\pm 0.376}$""","""${0.131}_{\pm 0.131}$"""
"""EnFF-F2P""",50,"""${0.286}_{\pm 0.135}$""","""${3.234}_{\pm 0.445}$""","""${0.521}_{\pm 0.028}$""","""${0.486}_{\pm 0.052}$""","""$\bm{{0.068}_{\pm 0.068}}$""","""$\bm{{0.333}_{\pm 0.333}}$""","""$\bm{{0.132}_{\pm 0.132}}$""","""$\bm{{0.066}_{\pm 0.066}}$"""
"""EnSF""",100,"""${0.605}_{\pm 0.215}$""","""$\bm{{5.648}_{\pm 0.522}}$""","""${1.098}_{\pm 0.044}$""","""${0.966}_{\pm 0.246}$""","""${0.199}_{\pm 0.199}$""","""${1.374}_{\pm 1.374}$""","""${0.900}_{\pm 0.900}$""","""${0.218}_{\pm 0.218}$"""


In [76]:
table_pl = table.pl()
table_pl_pivot = table_pl.pivot(
    on=['System'],
    index=['Model', '$T$'],
    values=['Time (s)', 'RMSE'],
    separator='|',
).fill_null(r'\texttt{NaN}')
cols = ['Model', '$T$']
for system in dataset_info['dataset_name_latex']:
    cols.extend([f'Time (s)|{system}', f'RMSE|{system}'])
table_pl_pivot = table_pl_pivot[cols]
gt = GT(table_pl_pivot)
gt = gt.tab_spanner(label=' ', columns=['Model', '$T$'])
# gt = gt.tab_stub(rowname_col='Model', groupname_col='$T$')
for system in dataset_info['dataset_name_latex']:
    cols = [f'Time (s)|{system}', f'RMSE|{system}']
    gt = gt.tab_spanner(label=system, columns=cols).cols_label(**{s: s.split('|')[0] for s in cols})
gt

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

GT(_tbl_data=shape: (15, 10)
┌──────────┬─────┬────────────┬────────────┬───┬────────────┬────────────┬────────────┬────────────┐
│ Model    ┆ $T$ ┆ Time       ┆ RMSE|Loren ┆ … ┆ Time (s)|N ┆ RMSE|Navie ┆ Time (s)|N ┆ RMSE|Navie │
│ ---      ┆ --- ┆ (s)|Lorenz ┆ z '96      ┆   ┆ avier-Stok ┆ r-Stokes   ┆ avier-Stok ┆ r-Stokes   │
│ str      ┆ i64 ┆ '96        ┆ ---        ┆   ┆ es ($64    ┆ ($64       ┆ es ($256   ┆ ($256      │
│          ┆     ┆ ---        ┆ str        ┆   ┆ \t…        ┆ \times…    ┆ \…         ┆ \time…     │
│          ┆     ┆ str        ┆            ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---        │
│          ┆     ┆            ┆            ┆   ┆ str        ┆ str        ┆ str        ┆ str        │
╞══════════╪═════╪════════════╪════════════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ EnSF     ┆ 5   ┆ $\bm{{0.26 ┆ ${7.674}_{ ┆ … ┆ \texttt{Na ┆ \texttt{Na ┆ \texttt{Na ┆ \texttt{Na │
│          ┆     ┆ 0}_{\pm    ┆ \pm        ┆   ┆ N}         ┆ N}         ┆ N}         ┆ N}         │
│          ┆     ┆ 0.010}}$   ┆ 7.674}$    ┆   ┆            ┆            ┆            ┆            │
│ EnFF-OT  ┆ 5   ┆ ${0.277}_{ ┆ $\bm{{0.62 ┆ … ┆ $\bm{{0.23 ┆ ${0.164}_{ ┆ $\bm{{0.12 ┆ ${0.495}_{ │
│          ┆     ┆ \pm        ┆ 4}_{\pm    ┆   ┆ 7}_{\pm    ┆ \pm        ┆ 4}_{\pm    ┆ \pm        │
│          ┆     ┆ 0.053}$    ┆ 0.624}}$   ┆   ┆ 0.017}}$   ┆ 0.164}$    ┆ 0.004}}$   ┆ 0.495}$    │
│ EnFF-F2P ┆ 5   ┆ ${0.273}_{ ┆ ${0.649}_{ ┆ … ┆ ${0.258}_{ ┆ $\bm{{0.06 ┆ ${0.125}_{ ┆ $\bm{{0.13 │
│          ┆     ┆ \pm        ┆ \pm        ┆   ┆ \pm        ┆ 7}_{\pm    ┆ \pm        ┆ 3}_{\pm    │
│          ┆     ┆ 0.049}$    ┆ 0.649}$    ┆   ┆ 0.033}$    ┆ 0.067}}$   ┆ 0.009}$    ┆ 0.133}}$   │
│ EnSF     ┆ 10  ┆ $\bm{{0.55 ┆ ${3.299}_{ ┆ … ┆ ${0.325}_{ ┆ ${1.560}_{ ┆ ${0.221}_{ ┆ ${5.405}_{ │
│          ┆     ┆ 3}_{\pm    ┆ \pm        ┆   ┆ \pm        ┆ \pm        ┆ \pm        ┆ \pm        │
│          ┆     ┆ 0.016}}$   ┆ 3.299}$    ┆   ┆ 0.025}$    ┆ 1.560}$    ┆ 0.076}$    ┆ 5.405}$    │
│ EnFF-OT  ┆ 10  ┆ ${0.581}_{ ┆ ${0.583}_{ ┆ … ┆ $\bm{{0.29 ┆ ${0.112}_{ ┆ ${0.210}_{ ┆ ${0.497}_{ │
│          ┆     ┆ \pm        ┆ \pm        ┆   ┆ 8}_{\pm    ┆ \pm        ┆ \pm        ┆ \pm        │
│          ┆     ┆ 0.109}$    ┆ 0.583}$    ┆   ┆ 0.025}}$   ┆ 0.112}$    ┆ 0.080}$    ┆ 0.497}$    │
│ …        ┆ …   ┆ …          ┆ …          ┆ … ┆ …          ┆ …          ┆ …          ┆ …          │
│ EnFF-OT  ┆ 50  ┆ ${3.133}_{ ┆ ${0.706}_{ ┆ … ┆ $\bm{{0.46 ┆ ${0.131}_{ ┆ $\bm{{0.51 ┆ ${0.376}_{ │
│          ┆     ┆ \pm        ┆ \pm        ┆   ┆ 3}_{\pm    ┆ \pm        ┆ 2}_{\pm    ┆ \pm        │
│          ┆     ┆ 0.500}$    ┆ 0.706}$    ┆   ┆ 0.049}}$   ┆ 0.131}$    ┆ 0.023}}$   ┆ 0.376}$    │
│ EnFF-F2P ┆ 50  ┆ ${3.234}_{ ┆ $\bm{{0.33 ┆ … ┆ ${0.486}_{ ┆ $\bm{{0.06 ┆ ${0.521}_{ ┆ $\bm{{0.13 │
│          ┆     ┆ \pm        ┆ 3}_{\pm    ┆   ┆ \pm        ┆ 6}_{\pm    ┆ \pm        ┆ 2}_{\pm    │
│          ┆     ┆ 0.445}$    ┆ 0.333}}$   ┆   ┆ 0.052}$    ┆ 0.066}}$   ┆ 0.028}$    ┆ 0.132}}$   │
│ EnSF     ┆ 100 ┆ $\bm{{5.64 ┆ ${1.374}_{ ┆ … ┆ ${0.966}_{ ┆ ${0.218}_{ ┆ ${1.098}_{ ┆ ${0.900}_{ │
│          ┆     ┆ 8}_{\pm    ┆ \pm        ┆   ┆ \pm        ┆ \pm        ┆ \pm        ┆ \pm        │
│          ┆     ┆ 0.522}}$   ┆ 1.374}$    ┆   ┆ 0.246}$    ┆ 0.218}$    ┆ 0.044}$    ┆ 0.900}$    │
│ EnFF-OT  ┆ 100 ┆ ${6.313}_{ ┆ ${0.721}_{ ┆ … ┆ \texttt{Na ┆ \texttt{Na ┆ $\bm{{0.93 ┆ ${0.271}_{ │
│          ┆     ┆ \pm        ┆ \pm        ┆   ┆ N}         ┆ N}         ┆ 8}_{\pm    ┆ \pm        │
│          ┆     ┆ 0.903}$    ┆ 0.721}$    ┆   ┆            ┆            ┆ 0.038}}$   ┆ 0.271}$    │
│ EnFF-F2P ┆ 100 ┆ ${6.435}_{ ┆ $\bm{{0.33 ┆ … ┆ $\bm{{0.96 ┆ $\bm{{0.06 ┆ ${1.006}_{ ┆ $\bm{{0.13 │
│          ┆     ┆ \pm        ┆ 4}_{\pm    ┆   ┆ 0}_{\pm    ┆ 6}_{\pm    ┆ \pm        ┆ 2}_{\pm    │
│          ┆     ┆ 0.764}$    ┆ 0.334}}$   ┆   ┆ 0.196}}$   ┆ 0.066}}$   ┆ 0.091}$    ┆ 0.132}}$   │
└──────────┴─────┴────────────┴─

In [81]:
table_latex = (
    gt.as_latex()
    .replace(r'\{', r'{')
    .replace(r'\}', r'}')
    .replace(r'\_', r'_')
    .replace(r'\$', r'$')
    .replace(r'\\pm', r'\pm')
    .replace(r'\\bm', r'\bm')
    .replace(r'\\times', r'\times')
    .replace(r'\\texttt', r'\texttt')
    .replace('Kuramoto-Sivashinsky', 'KS')
    .replace('Navier-Stokes', 'NS')
    .replace('EnFF-OT', r'EnFF-OT$^\ast$')
    .replace('EnFF-F2P', r'EnFF-F2P$^\ast$')
)
print(table_latex)

\begin{table}[!t]


\fontsize{12.0pt}{14.4pt}\selectfont

\begin{tabular*}{\linewidth}{@{\extracolsep{\fill}}lrllllllll}
\toprule
\multicolumn{2}{c}{ } & \multicolumn{2}{c}{Lorenz '96} & \multicolumn{2}{c}{KS (1,024 dim)} & \multicolumn{2}{c}{NS ($64 \times 64$)} & \multicolumn{2}{c}{NS ($256 \times 256$)} \\ 
\cmidrule(lr){1-2} \cmidrule(lr){3-4} \cmidrule(lr){5-6} \cmidrule(lr){7-8} \cmidrule(lr){9-10}
Model & $T$ & Time (s) & RMSE & Time (s) & RMSE & Time (s) & RMSE & Time (s) & RMSE \\ 
\midrule\addlinespace[2.5pt]
EnSF & 5 & $\bm{{0.260}_{\pm 0.010}}$ & ${7.674}_{\pm 7.674}$ & ${0.021}_{\pm 0.013}$ & ${1.630}_{\pm 1.630}$ & \texttt{NaN} & \texttt{NaN} & \texttt{NaN} & \texttt{NaN} \\
EnFF-OT$^\ast$ & 5 & ${0.277}_{\pm 0.053}$ & $\bm{{0.624}_{\pm 0.624}}$ & $\bm{{0.020}_{\pm 0.011}}$ & ${0.082}_{\pm 0.082}$ & $\bm{{0.237}_{\pm 0.017}}$ & ${0.164}_{\pm 0.164}$ & $\bm{{0.124}_{\pm 0.004}}$ & ${0.495}_{\pm 0.495}$ \\
EnFF-F2P$^\ast$ & 5 & ${0.273}_{\pm 0.049}$ & ${0.649}_{\pm 0.649}$ 